# Module 07 · Unit 01
# Counting Principles

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Adversarial Enumeration \[AE\] · Cryptographic Structure \[CS\] |
| **Refresher** | Module 03 · Unit 01 — Sets and Operations (cardinality) |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Apply the **multiplication rule** to count ordered sequences of independent choices
2. Apply the **addition rule** to count mutually exclusive cases
3. Apply **inclusion-exclusion** to count overlapping cases without double-counting
4. Compute **permutations** $P(n,k)$ and explain why order matters in security contexts
5. Compute **combinations** $\binom{n}{k}$ and apply them to attack surface sizing
6. State and apply the **Pigeonhole Principle** to collision resistance and token collisions
7. Compute the **birthday bound** and determine when a token or hash output space is too small


## 🔣 Symbol Primer

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $n!$ | Factorial | "$n$ factorial" | $n \times (n-1) \times \cdots \times 1$ — orderings of $n$ distinct items |
| $P(n,k)$ | Permutation | "P of $n$, $k$" | Ordered selections of $k$ items from $n$: $\frac{n!}{(n-k)!}$ |
| $\binom{n}{k}$ | Combination | "$n$ choose $k$" | Unordered selections of $k$ items from $n$: $\frac{n!}{k!(n-k)!}$ |
| $\lfloor x \rfloor$ | Floor | "floor of $x$" | Largest integer $\leq x$ |
| $\lceil x \rceil$ | Ceiling | "ceiling of $x$" | Smallest integer $\geq x$ |
| $\approx$ | Approximately | "approximately" | Close in value; used when exact form is unwieldy |

> **The connection to earlier modules.** Cardinality $|A|$ from Module 03
> counts the elements of a set. Combinatorics counts *arrangements* and
> *selections* — ordered and unordered — from those sets. The Pigeonhole
> Principle from Module 03 · Unit 03 reappears here as a formal theorem
> about injections, now applied to session tokens, hash outputs, and key spaces.

---


## 1 · The Multiplication Rule

If a task consists of $k$ independent steps, where step $i$ can be performed in
$n_i$ ways, the total number of ways to complete the task is:

$$n_1 \times n_2 \times \cdots \times n_k$$

This is the **multiplication rule** (also called the rule of product). It applies
whenever the choices at each step are independent — the number of ways to complete
step $i$ does not depend on which choices were made at earlier steps.

### Security Examples

**Password complexity.** A password policy requires exactly 8 characters, each
chosen from an alphabet of 94 printable ASCII characters (case-sensitive letters,
digits, symbols). The number of possible passwords is:

$$94^8 = 6{,}095{,}689{,}385{,}410{,}816 \approx 6 \times 10^{15}$$

At $10^9$ guesses per second (a fast GPU), this takes about 70 days of continuous
brute-force — marginal for a motivated attacker. With 12 characters: $94^{12} \approx
5 \times 10^{23}$, making brute force computationally infeasible for decades.

**API key space.** A 128-bit random API key has $2^{128}$ possible values —
approximately $3.4 \times 10^{38}$. This is the multiplication rule applied to
a sequence of 128 independent binary choices.

**The key insight:** adding one character to a password or one bit to a key
*multiplies* the search space. This is why length matters exponentially more
than character set size: a 12-character lowercase-only password has $26^{12}
\approx 9.5 \times 10^{16}$ combinations — more than an 8-character password
from a 94-character alphabet.


## 2 · Addition Rule and Inclusion-Exclusion

### Addition Rule

If a task can be completed by **exactly one** of $k$ mutually exclusive methods,
and method $i$ can be done in $n_i$ ways, the total count is:

$$n_1 + n_2 + \cdots + n_k$$

**Security example.** A penetration tester can enter a system through three
distinct entry vectors — web application (47 possible exploits), API endpoint
(12 exploits), or physical access (3 exploits). Total: $47 + 12 + 3 = 62$ entry paths.

### Inclusion-Exclusion

When cases overlap, the addition rule double-counts the intersection. The
**inclusion-exclusion principle** corrects this:

$$|A \cup B| = |A| + |B| - |A \cap B|$$

For three sets:

$$|A \cup B \cup C| = |A| + |B| + |C| - |A \cap B| - |A \cap C| - |B \cap C| + |A \cap B \cap C|$$

**Security example.** An organisation audits three access logs: web server ($|W| = 1{,}200$
requests), API server ($|A| = 800$ requests), database ($|D| = 400$ requests).
Some requests touched multiple systems. Without inclusion-exclusion, simply summing
gives $2{,}400$ — but this counts requests that appeared in two or three logs multiple
times. The actual number of distinct requests is:

$$|W \cup A \cup D| = 1200 + 800 + 400 - |W \cap A| - |W \cap D| - |A \cap D| + |W \cap A \cap D|$$

Inclusion-exclusion is the set-theoretic foundation of precise attack surface
measurement — without it, overlapping coverage leads to inflated counts.


## 3 · Permutations — When Order Matters

A **permutation** is an ordered arrangement. $P(n,k)$ counts the number of ways
to arrange $k$ items selected from a set of $n$ distinct items, where order matters:

$$P(n,k) = \frac{n!}{(n-k)!} = n \times (n-1) \times \cdots \times (n-k+1)$$

The special case $k = n$ gives $n!$ — the number of ways to arrange all $n$ items.

### Security Examples

**Attack step ordering.** An attacker has identified 6 exploitable vulnerabilities
and plans to use exactly 3 in sequence (each step depends on the previous). The
number of distinct 3-step attack chains is:

$$P(6,3) = 6 \times 5 \times 4 = 120$$

Each of the 120 orderings represents a different attack sequence that must be
considered in a threat model — because the same three vulnerabilities exploited
in different orders may produce different outcomes.

**Session token entropy.** A server generates session tokens as 6-character strings
drawn from 36 alphanumeric characters, without repetition. The number of possible
tokens is:

$$P(36,6) = 36 \times 35 \times 34 \times 33 \times 32 \times 31 = 1{,}402{,}410{,}240 \approx 1.4 \times 10^9$$

At $10^4$ requests per second, an attacker systematically enumerating tokens would
take about $1.4 \times 10^5$ seconds $\approx 39$ hours. This token space is dangerously
small for any production service — a reminder that "looks random" is not the same
as "has sufficient entropy."


## 4 · Combinations — When Order Does Not Matter

A **combination** counts unordered selections. $\binom{n}{k}$ (read "$n$ choose $k$")
gives the number of ways to select $k$ items from $n$ where order does not matter:

$$\binom{n}{k} = \frac{n!}{k!(n-k)!} = \frac{P(n,k)}{k!}$$

We divide by $k!$ because each unordered set of $k$ items corresponds to $k!$
ordered arrangements — all of which are the same combination.

### Pascal's Identity

$$\binom{n}{k} = \binom{n-1}{k-1} + \binom{n-1}{k}$$

A new item either is included in the selection (contribute $\binom{n-1}{k-1}$) or
is not (contribute $\binom{n-1}{k}$). Pascal's identity generates Pascal's triangle
and underlies the binomial theorem.

### Security Examples

**Vulnerability subset selection.** A red team has found 10 exploitable services.
They want to select 4 to test in the next sprint (order of testing doesn't matter
— they will all be tested). The number of possible selections is:

$$\binom{10}{4} = \frac{10!}{4! \cdot 6!} = 210$$

**Attack surface coverage.** A system has 20 input fields. An injection tester
wants to test every pair of fields simultaneously (combined injection). The number
of pairs is:

$$\binom{20}{2} = \frac{20!}{2! \cdot 18!} = 190$$

**Permission subset analysis.** A role has 8 available permissions. The number of
distinct permission sets (role configurations) is the power set size $2^8 = 256$
— which equals $\sum_{k=0}^{8} \binom{8}{k}$ by the binomial theorem. The
combination formula enumerates each level of the power set individually.


## 5 · The Pigeonhole Principle and the Birthday Bound

### Pigeonhole Principle (Formal Statement)

We first saw this in Module 03 · Unit 03 in the context of hash functions.
Here we state it precisely and apply it more broadly.

**Theorem.** If $n$ items are placed into $k < n$ containers, at least one
container must hold more than one item. Equivalently: if $|A| > |B|$, no
injective function $f: A \to B$ exists.

$$|A| > |B| \;\Rightarrow\; \nexists\, f: A \to B \text{ injective}$$

**Generalised form.** If $n$ items are placed into $k$ containers, at least
one container holds at least $\lceil n/k \rceil$ items.

### Security Applications

| Application | $A$ | $B$ | Consequence |
|---|---|---|---|
| Hash collisions | All possible inputs (infinite) | All $2^{256}$ hash outputs | Collisions mathematically guaranteed |
| Session token collision | All issued tokens over the service lifetime | Token space $|\mathcal{T}|$ | Collision once more than $|\mathcal{T}|$ tokens issued |
| Birthday attack on hash | Random inputs | Hash output space | Collision in $pprox \sqrt{|\mathcal{T}|}$ queries |
| IPv4 address exhaustion | All connected devices | $2^{32} pprox 4$B addresses | Address collision (NAT required) |

### The Birthday Bound

For a uniformly random function with output space of size $N$, the expected
number of evaluations before a collision is approximately:

$$E[\text{collision}] \approx \sqrt{\frac{\pi N}{2}} \approx 1.25 \sqrt{N}$$

The simpler engineering approximation used in practice:

$$\text{50\% collision probability after } \approx \sqrt{N} \text{ evaluations}$$

**Derivation sketch.** After $k$ draws, the probability of no collision is:

$$P(\text{no collision}) = 1 \cdot \frac{N-1}{N} \cdot \frac{N-2}{N} \cdots \frac{N-k+1}{N}
\approx e^{-k(k-1)/(2N)}$$

Setting this equal to $0.5$ and solving for $k$ gives $k \approx \sqrt{N \ln 4}
\approx 1.18\sqrt{N}$.

**The practical rule:** a token or hash output of $b$ bits is safe against
birthday attacks when $\sqrt{2^b} = 2^{b/2}$ exceeds the attacker's query budget.
For $b = 128$: the birthday bound is $2^{64} \approx 1.8 \times 10^{19}$ — infeasible.
For $b = 32$: $2^{16} \approx 65{,}536$ — feasible in seconds.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AE\] \[CS\]

| Counting concept | Security meaning | Rule of thumb |
|---|---|---|
| $n^k$ (multiplication) | Key space, password space, token space | Each extra character/bit multiplies the space |
| $P(n,k)$ (permutation) | Ordered attack chains; protocol message sequences | Order matters when steps have dependencies |
| $\binom{n}{k}$ (combination) | Vulnerability subsets; coverage selection | Order doesn't matter — just which $k$ from $n$ |
| $2^n$ (power set) | All possible permission sets for $n$ permissions | RBAC needed when $n > 20$ |
| Pigeonhole | Collision guarantee | If tokens exceed output space, collision is certain |
| Birthday bound $\sqrt{N}$ | Practical collision threshold | $b$-bit output safe when $2^{b/2} >$ attacker budget |

**The security sizing principle:** before deploying any system with a token,
nonce, session ID, or hash output, compute two numbers: (1) the total space size
$N$, and (2) the birthday bound $\sqrt{N}$. If the expected number of values
issued over the system's lifetime exceeds $\sqrt{N}$, collisions are likely.
If it exceeds $N$, collisions are certain.

---


## Python: Visualization & Verification

**1 · Counting Calculator** — implement the four core counting functions
(factorial, permutation, combination, power) and use them to compute security-relevant
quantities: password spaces, key spaces, attack chain counts.

**2 · Birthday Bound Simulator** — empirically verify the birthday bound by
running collision experiments at varying output sizes; compare experimental
results to the theoretical $\sqrt{N}$ prediction.

**3 · Attack Surface Sizing** — use combination counting to enumerate the attack
surface of a target system with multiple vulnerability classes; visualise how the
surface grows as the number of components scales.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from math import factorial, comb, perm, log2, sqrt, pi, exp
import random

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Counting Calculator — Security Space Sizes

We compute exact counts for a range of security-relevant scenarios, then
visualise the results on a log scale to make the orders-of-magnitude differences
visceral.


In [ ]:
# ── 1 · Counting Calculator ───────────────────────────────────────────────────

scenarios = [
    # (label, count, category)
    ("4-digit PIN (0-9)",             10**4,           'password'),
    ("6-char lowercase password",     26**6,           'password'),
    ("8-char mixed-case + digits",    62**8,           'password'),
    ("8-char full printable ASCII",   94**8,           'password'),
    ("12-char full printable ASCII",  94**12,          'password'),
    ("32-bit session token",          2**32,           'token'),
    ("64-bit session token",          2**64,           'token'),
    ("128-bit API key",               2**128,          'token'),
    ("DES key space (56-bit)",        2**56,           'crypto'),
    ("AES-128 key space",             2**128,          'crypto'),
    ("AES-256 key space",             2**256,          'crypto'),
    ("SHA-256 output space",          2**256,          'crypto'),
    ("3 exploits from 10 (ordered)",  perm(10, 3),     'attack'),
    ("3 exploits from 10 (any order)",comb(10, 3),     'attack'),
    ("5 vulns from 20 (any order)",   comb(20, 5),     'attack'),
]

cat_colors = {
    'password': TS_BLUE,
    'token':    TS_AMBER,
    'crypto':   TS_GREEN,
    'attack':   TS_RED,
}

print(f"{'Scenario':<40} {'Count':>30}  {'log₂':>8}  Birthday bound")
print("─" * 95)
for label, count, cat in scenarios:
    l2 = log2(count) if count > 0 else 0
    bday = f"2^{l2/2:.0f}" if l2 >= 2 else "N/A"
    count_str = f"{count:.3e}" if count > 1e12 else f"{count:,}"
    print(f"  {label:<38}  {count_str:>30}  {l2:>7.1f}  {bday}")

# ── Log-scale bar chart ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

labels = [s[0] for s in scenarios]
log_counts = [log2(s[1]) for s in scenarios]
colors = [cat_colors[s[2]] for s in scenarios]

bars = ax.barh(range(len(labels)), log_counts, color=colors,
               edgecolor='white', height=0.7)

# Annotate bars
for i, (lc, (label, count, cat)) in enumerate(zip(log_counts, scenarios)):
    ax.text(lc + 0.5, i, f"2^{lc:.0f}", va='center', fontsize=8, color=TS_GRAY)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('log₂(count) — each unit doubles the space')
ax.set_title('Security-Relevant Counting: Space Sizes on Log₂ Scale
'
             'Each additional unit = 2× harder to brute-force', pad=10, fontsize=11)

# Feasibility thresholds
ax.axvline(40, color=TS_RED, lw=1.5, linestyle='--', alpha=0.7,
           label='~2⁴⁰ ≈ 1T — feasible for well-funded attacker')
ax.axvline(80, color=TS_AMBER, lw=1.5, linestyle='--', alpha=0.7,
           label='~2⁸⁰ — marginal')
ax.axvline(128, color=TS_GREEN, lw=1.5, linestyle='--', alpha=0.7,
           label='~2¹²⁸ — infeasible with known technology')

legend = [
    mpatches.Patch(color=TS_BLUE,  label='Password space'),
    mpatches.Patch(color=TS_AMBER, label='Token space'),
    mpatches.Patch(color=TS_GREEN, label='Cryptographic space'),
    mpatches.Patch(color=TS_RED,   label='Attack chain count'),
]
ax.legend(handles=legend, loc='lower right', fontsize=9)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The log scale makes the staggering range of sizes visible on a single chart.
A few things stand out:

- The **4-digit PIN** (10,000 combinations) is so small it barely appears.
  At 10 guesses per second with lockout — common on ATMs — it takes 17 minutes
  worst case. Without lockout: under a second.
- **8-char ASCII passwords** sit below the 2⁴⁰ feasibility threshold — a
  well-funded attacker with specialised hardware can brute-force them.
- **12-char ASCII passwords** cross the infeasibility line. Three extra characters
  multiplied the space by $94^3 = 830{,}584$ — over three orders of magnitude.
- **AES-128 and SHA-256** share the same space size ($2^{128}$) — this is why
  128-bit security is the standard target, and why the birthday bound for SHA-256
  ($2^{128}$) is still infeasible despite the Pigeonhole Principle guaranteeing
  collisions exist.


### 2 · Birthday Bound Simulator

We empirically verify the birthday bound by running collision experiments at
multiple output space sizes. For each size $N$, we draw random values until a
collision occurs, repeat 1,000 times, and compare the mean collision point
against the theoretical prediction $\approx 1.25\sqrt{N}$.


In [ ]:
# ── 2 · Birthday Bound Simulator ─────────────────────────────────────────────
random.seed(42)

def birthday_experiment(N, trials=1000):
    """Run birthday experiment: draw from [0,N) until a collision. Repeat trials times."""
    collision_points = []
    for _ in range(trials):
        seen = set()
        count = 0
        while True:
            val = random.randrange(N)
            count += 1
            if val in seen:
                collision_points.append(count)
                break
            seen.add(val)
    return np.array(collision_points)

space_sizes = [100, 500, 1000, 5000, 10000, 50000]
results = {}

print(f"{'N (space size)':>16} {'Theory √N':>12} {'Experiment mean':>18} {'Ratio':>8}")
print("─" * 58)
for N in space_sizes:
    data = birthday_experiment(N, trials=2000)
    mean_exp  = data.mean()
    theory    = sqrt(N) * 1.25
    ratio     = mean_exp / theory
    results[N] = (data, mean_exp, theory)
    print(f"  {N:>14,}  {theory:>12.1f}  {mean_exp:>18.1f}  {ratio:>8.3f}")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: theory vs experiment
ax = axes[0]
Ns        = space_sizes
theories  = [sqrt(N) * 1.25 for N in Ns]
exp_means = [results[N][1] for N in Ns]

ax.plot(Ns, theories,  color=TS_BLUE,  lw=2.5, marker='o', label='Theory: $1.25\sqrt{N}$')
ax.plot(Ns, exp_means, color=TS_AMBER, lw=2.5, marker='s',
        linestyle='--', label='Experiment mean (2000 trials)')
ax.fill_between(Ns,
                [results[N][0].mean() - results[N][0].std() for N in Ns],
                [results[N][0].mean() + results[N][0].std() for N in Ns],
                alpha=0.15, color=TS_AMBER, label='±1 std dev')
ax.set_xlabel('Output space size $N$')
ax.set_ylabel('Draws until first collision')
ax.set_title('Birthday Bound: Theory vs Experiment', fontweight='bold', pad=8)
ax.legend(fontsize=9)

# Right: collision distribution histogram for N=10000
ax2   = axes[1]
data  = results[10000][0]
theory_10k = sqrt(10000) * 1.25
ax2.hist(data, bins=40, color=TS_BLUE, edgecolor='white', alpha=0.8, density=True)
ax2.axvline(theory_10k, color=TS_AMBER, lw=2.5, linestyle='--',
            label=f'Theory: {theory_10k:.0f}')
ax2.axvline(data.mean(), color=TS_GREEN, lw=2.5,
            label=f'Experiment mean: {data.mean():.0f}')
ax2.set_xlabel('Draws until first collision')
ax2.set_ylabel('Density')
ax2.set_title('Collision Distribution for N = 10,000', fontweight='bold', pad=8)
ax2.legend(fontsize=9)

fig.patch.set_facecolor('white')
plt.suptitle('Birthday Bound Verification — Theoretical vs Empirical',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**What do you see?**

The experimental means track the $1.25\sqrt{N}$ prediction closely across all
space sizes — the birthday bound is not a loose approximation, it is a tight
empirical law. The ratio column in the print output stays near 1.0 for all sizes.

The histogram for $N = 10{,}000$ shows the distribution of collision points —
it has a characteristic right-skewed shape. The 50% mark lands right at
$\sqrt{N} = 100$: half the experiments found a collision within the first 100
draws from a space of 10,000 values.

**The practical implication.** A service that issues 32-bit session tokens has
$N = 2^{32} \approx 4.3 \times 10^9$. The birthday bound is $\sqrt{N} \approx 65{,}536$.
After issuing 65,536 tokens, there is a 50% chance of a collision — two users
sharing the same session token. For a service with millions of users, this is
not a theoretical concern. It is a guarantee.


### 3 · Attack Surface Sizing

We model a target system with multiple vulnerability classes and use combination
counting to enumerate its attack surface: how many distinct ways can an attacker
select $k$ components to exploit simultaneously? We then show how the surface
grows as the number of components scales.


In [ ]:
# ── 3 · Attack Surface Sizing ────────────────────────────────────────────────

# Target system components
components = {
    'Web endpoints':     12,
    'API methods':       8,
    'Database tables':   15,
    'Admin functions':   5,
    'File upload points':3,
    'Auth mechanisms':   4,
    'Third-party libs':  22,
    'Config parameters': 18,
}

print("Attack Surface Analysis — Combination Counting")
print("=" * 60)
print(f"Total components: {sum(components.values())}")
print()

# Single-component attack (choose 1 from each class)
print("Single-component vulnerabilities (k=1):")
total_single = 0
for name, count in components.items():
    print(f"  {name:<25}  {count:>4} targets")
    total_single += count
print(f"  {'TOTAL':<25}  {total_single:>4}")
print()

# Multi-component attacks: choosing k from total pool
total_n = sum(components.values())
print(f"Multi-component attack surface (n = {total_n} total components):")
print(f"{'k (components)':>16} {'C(n,k)':>20}  {'Notes'}")
print("─" * 65)
for k in range(1, 8):
    c = comb(total_n, k)
    note = ('← single exploits' if k == 1 else
            '← pairs (e.g., CSRF+XSS)' if k == 2 else
            '← chains of 3' if k == 3 else
            '← complex multi-step attacks' if k == 4 else '')
    print(f"  {k:>14}  {c:>20,}  {note}")

# ── Growth visualisation ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: C(n,k) for fixed k=2,3,4 as n varies
ax = axes[0]
n_range = range(5, 51)
for k, color, label in [(2, TS_BLUE, 'k=2 (pairs)'),
                         (3, TS_AMBER, 'k=3 (triples)'),
                         (4, TS_GREEN, 'k=4 (quadruples)')]:
    ax.plot(n_range, [comb(n, k) for n in n_range],
            color=color, lw=2.5, label=label)

ax.axvline(total_n, color=TS_RED, lw=1.5, linestyle='--', alpha=0.7,
           label=f'Our system (n={total_n})')
ax.set_xlabel('Number of components $n$')
ax.set_ylabel('$\binom{n}{k}$ — number of $k$-component attack combinations')
ax.set_title('Attack Surface Growth with Component Count', fontweight='bold', pad=8)
ax.legend(fontsize=9)
ax.set_yscale('log')

# Right: surface breakdown by class (k=1)
ax2 = axes[1]
names = list(components.keys())
counts = list(components.values())
colors_bar = [TS_BLUE, TS_AMBER, TS_GREEN, TS_RED,
              '#8e44ad', '#1e8c8c', TS_GRAY, '#c0392b']

bars = ax2.bar(range(len(names)), counts, color=colors_bar[:len(names)],
               edgecolor='white', width=0.7)
for bar, c in zip(bars, counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             str(c), ha='center', fontsize=9, fontweight='bold')

ax2.set_xticks(range(len(names)))
ax2.set_xticklabels([n.replace(' ', '
') for n in names], fontsize=8)
ax2.set_ylabel('Number of individual attack targets')
ax2.set_title(f'Single-Component Attack Surface
(Total: {total_n} individual targets)',
              fontweight='bold', pad=8)

fig.patch.set_facecolor('white')
plt.suptitle('Attack Surface: Combination Counting Across Component Classes',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nKey insight: with {total_n} components, the number of 2-component")
print(f"attack pairs is C({total_n},2) = {comb(total_n,2):,} — each pair must")
print(f"be considered for combined exploitation (CSRF+stored XSS, etc.)")


**What do you see?**

The growth chart (left, log scale) shows how rapidly the multi-component attack
surface expands as the number of components increases. Even for $k = 2$ (pairs),
the surface grows quadratically. For $k = 3$ and $k = 4$, the growth is polynomial
in $n$ but the absolute numbers quickly become enormous.

For our system with 87 total components: there are $\binom{87}{2} = 3{,}741$ distinct
pairs to consider for combined exploitation — chained vulnerabilities, CSRF plus
stored XSS, race conditions requiring two simultaneous requests, and so on. No
manual penetration test can enumerate all of them; automated tools with combinatorial
coverage are the only practical approach.

The bar chart (right) shows which component classes contribute the most individual
targets. Third-party libraries and configuration parameters dominate — a reminder
that the supply chain and configuration surface are as large as the application
surface itself.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. INCLUSION-EXCLUSION IN PRACTICE
#    Three security scanners each report findings:
#      Scanner A:  142 findings
#      Scanner B:   98 findings
#      Scanner C:  115 findings
#    Overlaps: A∩B = 67, A∩C = 44, B∩C = 38, A∩B∩C = 29
#    Use inclusion-exclusion to compute the true number of distinct findings.
#    What fraction of findings does each scanner uniquely contribute?
#    Plot a Venn diagram using matplotlib patches.
#
# 2. OPTIMAL SESSION TOKEN LENGTH
#    A web service expects to issue at most 10 million active sessions
#    simultaneously. What is the minimum token length (in bits) such that
#    the probability of any collision among active tokens is below 0.1%?
#    (Hint: use the birthday approximation P(collision) ≈ k²/(2N) for small
#    probability, where k is the number of tokens and N is the token space.)
#    What token length (in bits) would you recommend for production use?
#
# 3. PERMUTATION VS COMBINATION IN ATTACK CHAINS
#    An attacker has identified 8 vulnerabilities in a web application.
#    For a 3-step attack chain:
#      (a) If each step must use a DIFFERENT vulnerability and ORDER MATTERS
#          (step 1 enables step 2): how many distinct chains are there?
#      (b) If each step must use a DIFFERENT vulnerability and ORDER DOESN'T MATTER:
#          how many distinct sets of 3?
#      (c) If the attacker can reuse the same vulnerability at multiple steps
#          and order matters: how many chains?
#    Verify your answers with Python calculations.

# Your work here:


---

## Summary

| Concept | Formula | Security application |
|---|---|---|
| **Multiplication rule** | $n_1 \times n_2 \times \cdots$ | Password spaces, key spaces, token spaces |
| **Addition rule** | $n_1 + n_2 + \cdots$ | Mutually exclusive attack vectors |
| **Inclusion-exclusion** | $|A \cup B| = |A|+|B|-|A \cap B|$ | Overlapping scanner findings; precise attack surface count |
| **Permutation** | $P(n,k) = \frac{n!}{(n-k)!}$ | Ordered attack chains; protocol message sequences |
| **Combination** | $\binom{n}{k} = \frac{n!}{k!(n-k)!}$ | Vulnerability subsets; coverage selection |
| **Pigeonhole** | $|A|>|B| \Rightarrow$ collision | Hash collisions unavoidable; token space exhaustion |
| **Birthday bound** | $\approx 1.25\sqrt{N}$ | Minimum token/hash bits for collision safety |

---

## Up Next

**Module 07 · Unit 02 — Adversarial Enumeration**

We apply the counting toolkit directly to AI security: LLM token vocabulary
sizes and jailbreak search spaces, adversarial example budgets in discrete feature
spaces, attack path enumeration in graphs, and the formal connection between
counting and the feasibility threshold that separates "theoretically possible"
from "practically achievable."

$\rightarrow$ `modules/module-07/unit-02-adversarial-enumeration.ipynb`
